# Carga dos arquivos raw (CSV) → Bronze

Lê **todos** os arquivos CSV do bucket `raw` (MinIO) com pandas e cria uma
tabela Iceberg por arquivo no schema `bronze_raw` (nome da tabela = nome do
arquivo sem a extensão). A escrita é idempotente (`createOrReplace`), então
reexecuções recriam as tabelas sem duplicar dados.

In [ ]:
SPARK_APP_NAME = "ingest-raw"

from nucleo import cria_spark_session

spark = cria_spark_session(SPARK_APP_NAME)
spark

In [ ]:
from os import environ as env

import pandas as pd
import s3fs

MINIO_ENDPOINT = env.get("MINIO_ENDPOINT", "http://minio:9000")
MINIO_ACCESS_KEY = env.get("MINIO_ACCESS_KEY", "unidata")
MINIO_SECRET_KEY = env.get("MINIO_SECRET_KEY", "supersenha")

SCHEMA = "bronze_raw"

spark.sql(f"CREATE NAMESPACE IF NOT EXISTS nessie.{SCHEMA}")

storage_options = {
    "key": MINIO_ACCESS_KEY,
    "secret": MINIO_SECRET_KEY,
    "client_kwargs": {"endpoint_url": MINIO_ENDPOINT},
}

fs = s3fs.S3FileSystem(**storage_options)
arquivos = sorted(f"s3://{p}" for p in fs.glob("s3://raw/*.csv"))
assert arquivos, f"nenhum arquivo CSV encontrado em s3://raw"

TABELAS = []
for arquivo in arquivos:
    df = pd.read_csv(arquivo, storage_options=storage_options)
    tabela = arquivo.rsplit("/", 1)[-1].rsplit(".", maxsplit=1)[0]
    spark.createDataFrame(df).writeTo(f"nessie.{SCHEMA}.{tabela}").createOrReplace()
    TABELAS.append(tabela)
    print(f"nessie.{SCHEMA}.{tabela}: {df.shape[0]} linhas")

In [ ]:
for tabela in TABELAS:
    n = spark.table(f"nessie.{SCHEMA}.{tabela}").count()
    print(f"{SCHEMA}.{tabela}: {n} linhas")